In [2]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    roc_curve
)

import matplotlib.pyplot as plt


# غيّر هذا المسار حسب مكان رفع فولدر processed_data في Google Drive
DATA_DIR = "/content/drive/MyDrive/processed_data2"

TRAIN_PATH = os.path.join(DATA_DIR, "train.csv")
TEST_PATH = os.path.join(DATA_DIR, "test.csv")

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

print("Train shape:", train.shape)
print("Test shape:", test.shape)

display(train.head())




# Prepare X/y
TARGET_COL = "isFraud"

X_train = train.drop(columns=[TARGET_COL])
y_train = train[TARGET_COL].astype(int)

X_test = test.drop(columns=[TARGET_COL])
y_test = test[TARGET_COL].astype(int)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

print("\nTrain target distribution:")
print(y_train.value_counts())

print("\nTest target distribution:")
print(y_test.value_counts())

Mounted at /content/drive
Train shape: (72270, 27)
Test shape: (1272524, 27)


,step,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,amount_log,balance_diff_orig,balance_diff_dest,amount_to_orig_balance,...,hour,day,amount_decile,type_encoded,type_CASH_IN,type_CASH_OUT,type_DEBIT,type_PAYMENT,type_TRANSFER,isFraud
0,0.938994,-0.297932,-0.271414,-0.275487,-0.323308,-0.333104,-2.646761,-0.147469,-0.153404,-0.139613,...,0.388547,0.928503,-1.566677,0.952340,-0.530843,-0.736581,-0.080961,1.399013,-0.302366,0
1,0.742253,0.143370,-0.288733,-0.292451,-0.156573,-0.081833,0.910125,-0.144618,0.286075,0.386275,...,-0.536923,0.759631,1.218720,-0.529067,-0.530843,1.357623,-0.080961,-0.714790,-0.302366,0
2,0.601725,-0.274859,-0.250979,-0.260057,-0.323308,-0.333104,-0.700605,-0.242287,-0.153404,-0.139612,...,0.388547,0.590758,-0.870328,0.952340,-0.530843,-0.736581,-0.080961,1.399013,-0.302366,0
3,0.601725,0.656136,-0.176202,-0.292451,-0.058214,0.068715,1.334531,-2.361123,0.555850,-0.139609,...,0.388547,0.590758,1.566894,-0.529067,-0.530843,1.357623,-0.080961,-0.714790,-0.302366,0
4,-1.386754,-0.289718,-0.281171,-0.286818,-0.323308,-0.333104,-1.241340,-0.181224,-0.153404,-0.139612,...,1.545386,-1.435712,-1.218503,0.952340,-0.530843,-0.736581,-0.080961,1.399013,-0.302366,0


X_train: (72270, 26)
X_test: (1272524, 26)

Train target distribution:
isFraud
0    65700
1     6570
Name: count, dtype: int64

Test target distribution:
isFraud
0    1270881
1       1643
Name: count, dtype: int64


In [ ]:
USE_TEST_SAMPLE = False
TEST_SAMPLE_SIZE = 100_000

if USE_TEST_SAMPLE:
    test_eval = test.sample(
        n=min(TEST_SAMPLE_SIZE, len(test)),
        random_state=42
    )
    X_eval = test_eval.drop(columns=[TARGET_COL])
    y_eval = test_eval[TARGET_COL].astype(int)
else:
    X_eval = X_test
    y_eval = y_test

print("Evaluation shape:", X_eval.shape)
print(y_eval.value_counts())

Evaluation shape: (1272524, 26)
isFraud
0    1270881
1       1643
Name: count, dtype: int64


In [4]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score
)

import pandas as pd
import time
import warnings

warnings.filterwarnings("ignore")

tree_results = []

def train_evaluate_tree_model(model, model_name):
    print("=" * 90)
    print(model_name)
    print(model)

    start_time = time.time()

    model.fit(X_train, y_train)

    y_pred = model.predict(X_eval)

    if hasattr(model, "predict_proba"):
        y_score = model.predict_proba(X_eval)[:, 1]
    else:
        y_score = y_pred

    total_time = time.time() - start_time

    print("\nClassification Report:")
    print(classification_report(y_eval, y_pred, digits=4))

    cm = confusion_matrix(y_eval, y_pred)
    print("Confusion Matrix:")
    print(cm)

    tn, fp, fn, tp = cm.ravel()

    accuracy = accuracy_score(y_eval, y_pred)
    precision_fraud = precision_score(y_eval, y_pred, pos_label=1, zero_division=0)
    recall_fraud = recall_score(y_eval, y_pred, pos_label=1, zero_division=0)
    f1_fraud = f1_score(y_eval, y_pred, pos_label=1, zero_division=0)
    roc_auc = roc_auc_score(y_eval, y_score)
    pr_auc = average_precision_score(y_eval, y_score)

    print(f"Accuracy: {accuracy:.8f}")
    print(f"ROC-AUC:  {roc_auc:.6f}")
    print(f"PR-AUC:   {pr_auc:.6f}")
    print(f"Time:     {total_time:.2f} seconds")

    tree_results.append({
        "model_name": model_name,
        "model_type": type(model).__name__,
        "n_estimators": getattr(model, "n_estimators", None),
        "max_depth": model.max_depth,
        "min_samples_split": model.min_samples_split,
        "min_samples_leaf": model.min_samples_leaf,
        "class_weight": str(model.class_weight),
        "accuracy": accuracy,
        "precision_fraud": precision_fraud,
        "recall_fraud": recall_fraud,
        "f1_fraud": f1_fraud,
        "roc_auc": roc_auc,
        "pr_auc": pr_auc,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp,
        "time_seconds": total_time
    })

    return model

In [6]:
# Model 1: Random Forest بدون class_weight - baseline
model1 = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    class_weight=None,
    n_jobs=-1,
    random_state=42
)

model1 = train_evaluate_tree_model(
    model1,
    "Model 1 - Random Forest - No class_weight - 100 trees"
)


# Model 2: Random Forest مع class_weight balanced
model2 = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    class_weight="balanced",
    n_jobs=-1,
    random_state=42
)

model2 = train_evaluate_tree_model(
    model2,
    "Model 2 - Random Forest - class_weight=balanced - 100 trees"
)


# Model 3: Random Forest مع class_weight balanced_subsample
model3 = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    class_weight="balanced_subsample",
    n_jobs=-1,
    random_state=42
)

model3 = train_evaluate_tree_model(
    model3,
    "Model 3 - Random Forest - class_weight=balanced_subsample - 100 trees"
)


# Model 4: Random Forest بدون class_weight مع تقييد العمق
model4 = RandomForestClassifier(
    n_estimators=200,
    max_depth=20,
    min_samples_split=2,
    min_samples_leaf=1,
    class_weight=None,
    n_jobs=-1,
    random_state=42
)

model4 = train_evaluate_tree_model(
    model4,
    "Model 4 - Random Forest - No class_weight - max_depth=20 - 200 trees"
)


# Model 5: Random Forest مع class_weight balanced و max_depth=20
model5 = RandomForestClassifier(
    n_estimators=200,
    max_depth=20,
    min_samples_split=2,
    min_samples_leaf=1,
    class_weight="balanced",
    n_jobs=-1,
    random_state=42
)

model5 = train_evaluate_tree_model(
    model5,
    "Model 5 - Random Forest - balanced - max_depth=20 - 200 trees"
)

Model 1 - Random Forest - No class_weight - 100 trees
RandomForestClassifier(n_jobs=-1, random_state=42)

Classification Report:
              precision    recall  f1-score   support

           0     1.0000    1.0000    1.0000   1270881
           1     0.9832    0.9976    0.9903      1643

    accuracy                         1.0000   1272524
   macro avg     0.9916    0.9988    0.9952   1272524
weighted avg     1.0000    1.0000    1.0000   1272524

Confusion Matrix:
[[1270853      28]
 [      4    1639]]
Accuracy: 0.99997485
ROC-AUC:  0.999381
PR-AUC:   0.998699
Time:     34.85 seconds
Model 2 - Random Forest - class_weight=balanced - 100 trees
RandomForestClassifier(class_weight='balanced', n_jobs=-1, random_state=42)

Classification Report:
              precision    recall  f1-score   support

           0     1.0000    1.0000    1.0000   1270881
           1     0.9704    0.9976    0.9838      1643

    accuracy                         1.0000   1272524
   macro avg     0.9852   

In [7]:
# Model 6: Decision Tree بدون class_weight - baseline
model6 = DecisionTreeClassifier(
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    class_weight=None,
    random_state=42
)

model6 = train_evaluate_tree_model(
    model6,
    "Model 6 - Decision Tree - No class_weight"
)


# Model 7: Decision Tree مع class_weight balanced
model7 = DecisionTreeClassifier(
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    class_weight="balanced",
    random_state=42
)

model7 = train_evaluate_tree_model(
    model7,
    "Model 7 - Decision Tree - class_weight=balanced"
)


# Model 8: Decision Tree بدون class_weight مع max_depth=10
model8 = DecisionTreeClassifier(
    max_depth=10,
    min_samples_split=2,
    min_samples_leaf=1,
    class_weight=None,
    random_state=42
)

model8 = train_evaluate_tree_model(
    model8,
    "Model 8 - Decision Tree - No class_weight - max_depth=10"
)


# Model 9: Decision Tree مع class_weight balanced و max_depth=10
model9 = DecisionTreeClassifier(
    max_depth=10,
    min_samples_split=2,
    min_samples_leaf=1,
    class_weight="balanced",
    random_state=42
)

model9 = train_evaluate_tree_model(
    model9,
    "Model 9 - Decision Tree - balanced - max_depth=10"
)


# Model 10: Decision Tree أكثر تحفظًا لتقليل overfitting
model10 = DecisionTreeClassifier(
    max_depth=15,
    min_samples_split=10,
    min_samples_leaf=5,
    class_weight=None,
    random_state=42
)

model10 = train_evaluate_tree_model(
    model10,
    "Model 10 - Decision Tree - No class_weight - regularized"
)


# Model 11: Decision Tree أكثر تحفظًا مع class_weight
model11 = DecisionTreeClassifier(
    max_depth=15,
    min_samples_split=10,
    min_samples_leaf=5,
    class_weight="balanced",
    random_state=42
)

model11 = train_evaluate_tree_model(
    model11,
    "Model 11 - Decision Tree - balanced - regularized"
)

Model 6 - Decision Tree - No class_weight
DecisionTreeClassifier(random_state=42)

Classification Report:
              precision    recall  f1-score   support

           0     1.0000    0.9996    0.9998   1270881
           1     0.7684    0.9976    0.8681      1643

    accuracy                         0.9996   1272524
   macro avg     0.8842    0.9986    0.9340   1272524
weighted avg     0.9997    0.9996    0.9996   1272524

Confusion Matrix:
[[1270387     494]
 [      4    1639]]
Accuracy: 0.99960865
ROC-AUC:  0.998588
PR-AUC:   0.766534
Time:     1.57 seconds
Model 7 - Decision Tree - class_weight=balanced
DecisionTreeClassifier(class_weight='balanced', random_state=42)

Classification Report:
              precision    recall  f1-score   support

           0     1.0000    0.9996    0.9998   1270881
           1     0.7843    0.9982    0.8784      1643

    accuracy                         0.9996   1272524
   macro avg     0.8922    0.9989    0.9391   1272524
weighted avg     0.

In [8]:
tree_results_df = pd.DataFrame(tree_results)

display(
    tree_results_df.sort_values(
        by=["f1_fraud", "precision_fraud", "recall_fraud"],
        ascending=False
    )
)

,model_name,model_type,n_estimators,max_depth,min_samples_split,min_samples_leaf,class_weight,accuracy,precision_fraud,recall_fraud,f1_fraud,roc_auc,pr_auc,TN,FP,FN,TP,time_seconds
3,Model 4 - Random Forest - No class_weight - ma...,RandomForestClassifier,200.0,20.0,2,1,None,0.999980,0.986755,0.997565,0.992131,0.999374,0.998697,1270859,22,4,1639,52.382193
0,Model 1 - Random Forest - No class_weight - 10...,RandomForestClassifier,100.0,NaN,2,1,None,0.999975,0.983203,0.997565,0.990332,0.999381,0.998699,1270853,28,4,1639,34.845757
4,Model 5 - Random Forest - balanced - max_depth...,RandomForestClassifier,200.0,20.0,2,1,balanced,0.999972,0.980850,0.997565,0.989137,0.999376,0.998708,1270849,32,4,1639,52.526419
2,Model 3 - Random Forest - class_weight=balance...,RandomForestClassifier,100.0,NaN,2,1,balanced_subsample,0.999964,0.975015,0.997565,0.986161,0.999382,0.998677,1270839,42,4,1639,28.592871
1,Model 2 - Random Forest - class_weight=balance...,RandomForestClassifier,100.0,NaN,2,1,balanced,0.999958,0.970397,0.997565,0.983794,0.999381,0.998714,1270831,50,4,1639,26.628146
9,Model 10 - Decision Tree - No class_weight - r...,DecisionTreeClassifier,NaN,15.0,10,5,None,0.999913,0.937714,0.998783,0.967286,0.999389,0.996886,1270772,109,2,1641,1.601129
7,Model 8 - Decision Tree - No class_weight - ma...,DecisionTreeClassifier,NaN,10.0,2,1,None,0.999857,0.902036,0.997565,0.947399,0.999559,0.900866,1270703,178,4,1639,1.870792
8,Model 9 - Decision Tree - balanced - max_depth=10,DecisionTreeClassifier,NaN,10.0,2,1,balanced,0.999726,0.825453,0.998783,0.903883,0.999575,0.922179,1270534,347,2,1641,2.273414
6,Model 7 - Decision Tree - class_weight=balanced,DecisionTreeClassifier,NaN,NaN,2,1,balanced,0.999643,0.784314,0.998174,0.878415,0.998910,0.782884,1270430,451,3,1640,1.695534
5,Model 6 - Decision Tree - No class_weight,DecisionTreeClassifier,NaN,NaN,2,1,None,0.999609,0.768401,0.997565,0.868114,0.998588,0.766534,1270387,494,4,1639,1.573510
